# Session 5: a multiplier from gates
**A. Real-world problem.** The adder of Session 3 can add and subtract, but a processor also has
to multiply: scaling a sound, mixing colours, computing an address. There is no multiply gate.
Yet two 4-bit numbers can be multiplied with nothing more than AND gates and the half and full
adders we already have. This session builds that multiplier in the gate simulator, tests it on
all 256 input pairs against Python's own `*`, and then asks what it costs: how many gates,
how long the slowest product bit takes, and why the answer needs twice as many bits as the
inputs.

**Plan.** (1) multiplying on paper in binary, (2) partial products from AND gates, (3) adding
the rows with half and full adders, (4) the multiplier and its check, (5) size, delay and
product width, (6) see it.
This is a simplified educational model for unsigned numbers: ideal gates with a delay of 1 unit
each, no wires, no clock, and no real chip timing.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the simulator and the adder cells (from Session 3)
A circuit is a list of gates; each wire has a *depth*, the gate delays on its longest path. A half
adder (XOR and AND) adds two bits, and a full adder (two half adders and an OR) adds two bits and
a carry.

In [ ]:
OPS = {
    "AND": lambda a, b: a & b,
    "OR": lambda a, b: a | b,
    "XOR": lambda a, b: a ^ b,
    "NAND": lambda a, b: 1 - (a & b),
    "NOT": lambda a: 1 - a,
}


class Circuit:
    def __init__(self):
        self.gates = []          # (output wire, op, input wires) in the order they were added
        self.depth = {}          # wire name -> gate delays from the inputs

    def input(self, name):
        self.depth[name] = 0
        return name

    def gate(self, op, *ins):
        out = f"w{len(self.gates)}"
        self.gates.append((out, op, ins))
        self.depth[out] = 1 + max(self.depth[i] for i in ins)
        return out

    def run(self, values):
        """values: dict input name -> array of 0/1. Returns every wire's array."""
        wires = {k: np.asarray(v, dtype=np.int64) for k, v in values.items()}
        for out, op, ins in self.gates:
            wires[out] = OPS[op](*[wires[i] for i in ins])
        return wires

    def size(self):
        return len(self.gates)

    def mix(self):
        """How many gates of each kind: {'AND': 36, ...}."""
        out = {}
        for _, op, _ in self.gates:
            out[op] = out.get(op, 0) + 1
        return out


def half_adder(c, a, b):
    return c.gate("XOR", a, b), c.gate("AND", a, b)          # sum, carry


def full_adder(c, a, b, cin):
    s1, c1 = half_adder(c, a, b)
    s, c2 = half_adder(c, s1, cin)
    return s, c.gate("OR", c1, c2)                           # sum, carry out


def bits_of(values, n):
    """Array of unsigned numbers -> list of n arrays (bit 0 first)."""
    values = np.asarray(values, dtype=np.int64)
    return [(values >> i) & 1 for i in range(n)]


def number_of(bit_arrays):
    """List of bit arrays (bit 0 first) -> array of unsigned numbers."""
    return sum(b.astype(np.int64) << i for i, b in enumerate(bit_arrays))


def ripple_adder_stats(n):
    """Gates and carry-out delay of the Session 3 ripple adder with n bits (for comparison)."""
    c = Circuit()
    a = [c.input(f"a{i}") for i in range(n)]
    b = [c.input(f"b{i}") for i in range(n)]
    carry = c.input("cin")
    for i in range(n):
        _, carry = full_adder(c, a[i], b[i], carry)
    return c.size(), c.depth[carry]

## Step 2: multiplying on paper, in binary
Multiplying 13 x 11 on paper, you multiply 13 by each digit of 11 and add the rows, each shifted
one place further left. In binary every digit is 0 or 1, so each row is either 0 or a copy of the
first number, shifted. These rows are the **partial products**:

      1101      13 x b0 (b0 = 1)
     11010      13 x b1 (b1 = 1), shifted 1
    000000      13 x b2 (b2 = 0), shifted 2
   1101000      13 x b3 (b3 = 1), shifted 3
   --------
  10001111      = 143

Column by column: add the ones in the column and the carry from the column before, write the
total's last bit and carry the rest to the next column.

In [ ]:
def partial_products(a, b, n):
    """The n partial products of a x b: row j is a shifted left j places if bit j of b is 1, else 0."""
    return [(a << j) if (b >> j) & 1 else 0 for j in range(n)]


def shift_and_add(a, b, n):
    """Multiply by adding the partial products."""
    return sum(partial_products(a, b, n))


def column_addition(a, b, n):
    """Add the partial products column by column, as on paper.

    Returns a list for columns 0 to 2n - 1 of dicts with: ones (partial-product bits that are 1 in
    this column), carry_in, total, bit (the product bit) and carry_out.
    """
    cols, carry = [], 0
    for k in range(2 * n):
        ones = sum(((a >> i) & 1) & ((b >> (k - i)) & 1) for i in range(n) if 0 <= k - i < n)
        total = ones + carry
        cols.append({"ones": ones, "carry_in": carry, "total": total, "bit": total % 2, "carry_out": total // 2})
        carry = total // 2
    return cols

## Step 3: the partial products are AND gates
Bit i of row j is `a_i AND b_j`: it is 1 only when both bits are 1. A 4 x 4 multiplier needs
16 AND gates, one for every cell of the paper layout, all working at once (depth 1).

## Step 4: add the rows with half and full adders
We add the rows one at a time. In a column where two bits meet we use a half adder; where two
bits and a carry meet we use a full adder; a lone bit passes straight through. The carry out of
one cell goes to the next column of the same row. This regular grid of cells is called an
**array multiplier**.

In [ ]:
def partial_product_wires(c, a, b):
    """AND gates: pp[j][i] = a_i AND b_j."""
    return [[c.gate("AND", ai, bj) for ai in a] for bj in b]


def add_column(c, signals):
    """Add up to three bits in one column: returns (sum wire, carry wire or None). None means 'no wire'."""
    sigs = [s for s in signals if s is not None]
    if len(sigs) == 1:
        return sigs[0], None
    if len(sigs) == 2:
        return half_adder(c, *sigs)
    return full_adder(c, *sigs)


def build_multiplier(n):
    """An n x n unsigned array multiplier (n >= 2). Returns a dict describing it; 'product' has 2n wires."""
    c = Circuit()
    a = [c.input(f"a{i}") for i in range(n)]
    b = [c.input(f"b{j}") for j in range(n)]
    pp = partial_product_wires(c, a, b)
    acc = [None] * (2 * n)
    for i in range(n):
        acc[i] = pp[0][i]
    for j in range(1, n):
        carry = None
        for i in range(n):
            acc[j + i], carry = add_column(c, [acc[j + i], pp[j][i], carry])
        acc[j + n] = carry
    return {"c": c, "n": n, "a": a, "b": b, "product": acc}


def multiply(mult, a, b):
    """Run a built multiplier on arrays of unsigned numbers; returns the products."""
    n = mult["n"]
    a, b = np.broadcast_arrays(np.asarray(a), np.asarray(b))
    vals = {}
    for i, (x, y) in enumerate(zip(bits_of(a, n), bits_of(b, n))):
        vals[mult["a"][i]], vals[mult["b"][i]] = x, y
    w = mult["c"].run(vals)
    return number_of([w[p] for p in mult["product"]])


def mismatches(mult, trials=20000, seed=5):
    """Number of input pairs where the multiplier disagrees with Python's a * b.

    Every pair is tried for n <= 8; wider multipliers get random pairs plus the extremes.
    """
    n = mult["n"]
    if n <= 8:
        A, B = np.meshgrid(np.arange(2 ** n), np.arange(2 ** n))
        A, B = A.ravel(), B.ravel()
    else:
        rng = np.random.default_rng(seed)
        top = 2 ** n - 1
        A = np.concatenate([rng.integers(0, 2 ** n, trials), [0, top, top, 1, top]])
        B = np.concatenate([rng.integers(0, 2 ** n, trials), [top, 0, top, top, 1]])
    return int(np.sum(multiply(mult, A, B) != A * B))

## Step 5: size, delay and the width of the answer
We can count the gates and read the delay of every product bit from the simulator. Each
of the n rows after the first needs n adder cells, so the gate count grows with n squared, while
the longest path runs down through the rows, so the delay grows in step with n. An n-bit number
is at most 2^n - 1, so the product is at most (2^n - 1)^2 < 2^(2n): it needs 2n bits and can
never overflow if you keep them all. Keep only the low n bits, as a narrow chip might, and
most answers are wrong.

In [ ]:
def gate_count_formula(n):
    """6n^2 - 8n: n^2 AND gates, n half adders (2 gates) and n(n - 2) full adders (5 gates)."""
    return 6 * n * n - 8 * n


def product_delays(mult):
    """Gate delays until each product bit is valid, bit 0 first."""
    return [mult["c"].depth[p] for p in mult["product"]]


def design_row(n):
    m = build_multiplier(n)
    return {"n": n, "gates": m["c"].size(), "depth": max(product_delays(m)), "bits": len(m["product"])}


def design_table(widths=range(2, 11)):
    return [design_row(n) for n in widths]


def widest_within(max_delay, limit=16):
    """Widest n (2 to limit) whose slowest product bit is ready within max_delay gate delays."""
    return max((n for n in range(2, limit + 1) if design_row(n)["depth"] <= max_delay), default=0)


def low_half(product, n):
    """What a multiplier that keeps only n bits would return."""
    return product % (2 ** n)


def time_ns(depth, gate_ns=0.5):
    return depth * gate_ns

## Step 6: see it
Left: all 256 products of two 4-bit numbers (a down the side, b along the bottom): symmetric,
zero along the first row and column, and up to 225 at the corner. Right: the gates and the
delay of the array multiplier against n, read from the simulator.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m4 = build_multiplier(4)
    A, B = np.meshgrid(np.arange(16), np.arange(16))
    P = multiply(m4, A.ravel(), B.ravel()).reshape(16, 16)
    print(f"4 x 4 multiplier: {m4['c'].size()} gates {m4['c'].mix()}, product-bit delays {product_delays(m4)}")
    print("all 256 pairs match Python:", mismatches(m4) == 0)
    fig, ax = plt.subplots(figsize=(5.8, 4.8))
    im = ax.imshow(P, cmap="YlOrBr", origin="lower", extent=(-0.5, 15.5, -0.5, 15.5))
    ax.set_xlabel("b (4 bits)")
    ax.set_ylabel("a (4 bits)")
    ax.set_title("Product from the gate-level multiplier")
    fig.colorbar(im, ax=ax, label="a x b (8-bit product)")
    fig.tight_layout()
    fig.savefig(OUT / "product_table.png", dpi=120)

    rows = design_table()
    ns = [r["n"] for r in rows]
    fig2, axs = plt.subplots(1, 2, figsize=(9.6, 3.8))
    axs[0].plot(ns, [r["gates"] for r in rows], "o-", color="#0072b2")
    axs[0].set_xlabel("multiplier width n (bits)")
    axs[0].set_ylabel("gates")
    axs[0].set_title("Size grows as n squared (6n^2 - 8n)")
    axs[1].plot(ns, [r["depth"] for r in rows], "s--", color="#c0561b")
    axs[1].axhline(20, color="#8a1a5b", ls=":", label="10 ns budget (20 gate delays)")
    axs[1].set_xlabel("multiplier width n (bits)")
    axs[1].set_ylabel("slowest product bit (gate delays)")
    axs[1].set_title("Delay grows in step with n (6n - 8)")
    axs[1].legend()
    for a_ in axs:
        a_.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "size_delay.png", dpi=120)
    for r in rows:
        print(f"n = {r['n']:2d}: {r['gates']:4d} gates, slowest product bit after {r['depth']:2d} gate delays")
    print("7 x 3 columns:", [(c["ones"], c["carry_in"], c["bit"]) for c in column_addition(7, 3, 4)][:5])
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print the gate count and the slowest product-bit delay of `build_multiplier(n)` for n = 2, 3, 4. Predict each from 6n^2 - 8n and 6n - 8 first (n = 2 is a special case).
2. **Stretch.** Use `multiply` on all 256 pairs of 4-bit numbers and list every pair whose product is 36. Check your list against the divisors of 36 that are below 16.
3. **Explain.** Stretch: a multiplier that keeps only the low 4 bits gives the wrong answer for most pairs. Use `low_half` to count the pairs of 4-bit numbers where it is still right, and explain why the count is 76.

## Assessment
- **A. Conceptual.** Why does the product of two 4-bit numbers need 8 bits, and why does multiplying by 8 need no adder at all?
- **B. Numerical.** Multiply 13 x 11 by partial products: write each row, add them, and give the product in binary. Sanity check with decimal arithmetic.
- **C. Graph and simulation interpretation.** From the size and delay graph, how many gates and how many gate delays does the multiplier have at n = 4 and at n = 8? What happens to each when the width doubles?
- **D. Design and optimisation challenge.** Design: each gate takes 0.5 ns and the product of an array multiplier must be ready within 10 ns. What is the widest multiplier allowed? What would you change to go wider?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # multiplying on paper
    checks.append(("13 x 11: the partial products are 13, 26, 0 and 104 and they add up to 143 = 10001111",
                   L.partial_products(13, 11, 4) == [13, 26, 0, 104] and sum(L.partial_products(13, 11, 4)) == 143
                   and format(143, "08b") == "10001111"))
    A, B = np.meshgrid(np.arange(16), np.arange(16))
    A, B = A.ravel(), B.ravel()
    checks.append(("adding the partial products gives a x b for all 256 pairs of 4-bit numbers",
                   all(L.shift_and_add(int(a), int(b), 4) == int(a) * int(b) for a, b in zip(A, B))))
    checks.append(("each partial product is 0 or a copy of a shifted: row j is a << j when bit j of b is 1",
                   all(L.partial_products(a, b, 4)[j] == (a << j if (b >> j) & 1 else 0)
                       for a in range(16) for b in range(16) for j in range(4))))
    cols = L.column_addition(7, 3, 4)
    checks.append(("worked example 7 x 3 by columns: ones 1, 2, 2, 1, 0 and carries 0, 0, 1, 1, 1 give product bits 1, 0, 1, 0, 1 = 10101 = 21",
                   [c["ones"] for c in cols[:5]] == [1, 2, 2, 1, 0] and [c["carry_in"] for c in cols[:5]] == [0, 0, 1, 1, 1]
                   and sum(c["bit"] << k for k, c in enumerate(cols)) == 21 == 0b10101))
    ok = all(sum(c["bit"] << k for k, c in enumerate(L.column_addition(int(a), int(b), 4))) == int(a) * int(b) for a, b in zip(A, B))
    checks.append(("column addition (write the last bit, carry the rest) gives the product for all 256 pairs", bool(ok)))
    mx = max(c["carry_out"] for a, b in zip(A, B) for c in L.column_addition(int(a), int(b), 4))
    checks.append(("a carry from one column can be bigger than 1: the largest carry in a 4 x 4 product is 3", mx == 3))

    # the gate-level multiplier
    m4 = L.build_multiplier(4)
    checks.append(("4 x 4 multiplier: every one of the 256 input pairs matches Python's a * b (roadmap checkpoint)",
                   L.mismatches(m4) == 0 and np.array_equal(L.multiply(m4, A, B), A * B)))
    checks.append(("the multiplier has 8 product wires, and the largest product 15 x 15 = 225 = 11100001",
                   len(m4["product"]) == 8 and int(L.multiply(m4, 15, 15)) == 225 == 0b11100001))
    allgood = all(L.mismatches(L.build_multiplier(n)) == 0 for n in (2, 3, 5, 6, 7, 8))
    checks.append(("the same design is right for every pair at n = 2, 3, 5, 6, 7 and 8 (8 x 8 is 65536 pairs)", bool(allgood)))
    checks.append(("a 12-bit multiplier matches Python on 20000 random pairs and the extremes",
                   L.mismatches(L.build_multiplier(12)) == 0))
    broken = dict(m4, product=m4["product"][:7] + [m4["product"][6]])
    checks.append(("a wrong multiplier is caught: copying product bit 6 into bit 7 gives mismatches", L.mismatches(broken) > 0))
    checks.append(("worked examples through the gates: 7 x 3 = 21, 13 x 11 = 143, 6 x 5 = 30",
                   int(L.multiply(m4, 7, 3)) == 21 and int(L.multiply(m4, 13, 11)) == 143 and int(L.multiply(m4, 6, 5)) == 30))

    # properties of the circuit
    P = L.multiply(m4, A, B)
    checks.append(("multiplication is commutative in the circuit too: a x b = b x a on all 256 pairs",
                   np.array_equal(P.reshape(16, 16), P.reshape(16, 16).T)))
    checks.append(("multiplying by 1, 2, 4 or 8 is a shift: the circuit gives a, 2a, 4a, 8a for every a",
                   all(np.array_equal(L.multiply(m4, np.arange(16), b), np.arange(16) << sh) for b, sh in ((1, 0), (2, 1), (4, 2), (8, 3)))))
    checks.append(("31 of the 256 products are zero (a = 0 or b = 0) and 6 equal 12",
                   int((P == 0).sum()) == 31 and int((P == 12).sum()) == 6))

    # size
    checks.append(("4 x 4 multiplier: 64 gates = 36 AND + 20 XOR + 8 OR", m4["c"].size() == 64 and m4["c"].mix() == {"AND": 36, "XOR": 20, "OR": 8}))
    checks.append(("the partial products take 16 AND gates, all ready after one gate delay",
                   sum(1 for _, op, ins in m4["c"].gates[:16] if op == "AND") == 16
                   and all(m4["c"].depth[g[0]] == 1 for g in m4["c"].gates[:16])))
    checks.append(("gate count = 6n^2 - 8n for every width n = 2 to 12 (8: 320 gates)",
                   all(L.build_multiplier(n)["c"].size() == L.gate_count_formula(n) for n in range(2, 13))
                   and L.gate_count_formula(8) == 320))
    ok = True
    for n in range(2, 11):
        mix = L.build_multiplier(n)["c"].mix()
        ok &= mix.get("OR", 0) == n * (n - 2) and mix["XOR"] == n + 2 * n * (n - 2) and mix["AND"] == n * n + n + 2 * n * (n - 2)
    checks.append(("n x n multiplier: n^2 + n + 2n(n-2) AND gates, n + 2n(n-2) XOR and n(n-2) OR gates (n half adders, n(n-2) full adders)", bool(ok)))

    # delay
    checks.append(("4 x 4: product bit delays are 1, 2, 4, 8, 11, 13, 15, 16 gate delays",
                   L.product_delays(m4) == [1, 2, 4, 8, 11, 13, 15, 16]))
    checks.append(("slowest product bit: 16 gate delays at 4 bits and 40 at 8 bits",
                   max(L.product_delays(m4)) == 16 and max(L.product_delays(L.build_multiplier(8))) == 40))
    checks.append(("slowest product bit = 6n - 8 gate delays for n = 3 to 16 (n = 2 gives 3)",
                   all(L.design_row(n)["depth"] == 6 * n - 8 for n in range(3, 17)) and L.design_row(2)["depth"] == 3))
    checks.append(("times at 0.5 ns per gate delay: 4 x 4 takes 8 ns, 8 x 8 takes 20 ns", L.time_ns(16) == 8.0 and L.time_ns(40) == 20.0))

    # compared with the adder of Session 3
    g8, d8 = L.ripple_adder_stats(8)
    checks.append(("an 8-bit ripple adder has 40 gates and 17 gate delays; the 8 x 8 multiplier has 8 times the gates (320) and more than twice the delay (40)",
                   (g8, d8) == (40, 17) and 320 // g8 == 8 and 40 / d8 > 2))
    r4, r8 = L.design_row(4), L.design_row(8)
    checks.append(("doubling the width from 4 to 8 multiplies the gates by 5 (64 to 320) and the delay by 2.5 (16 to 40)",
                   r8["gates"] / r4["gates"] == 5.0 and r8["depth"] / r4["depth"] == 2.5))

    # width of the answer
    checks.append(("an n-bit product needs 2n bits: (2^n - 1)^2 < 2^(2n) for n = 2 to 12, and 15 x 15 = 225 needs all 8 bits of the 4 x 4 product",
                   all((2 ** n - 1) ** 2 < 2 ** (2 * n) for n in range(2, 13)) and 225 >= 128
                   and all(L.design_row(n)["bits"] == 2 * n for n in range(2, 11))))
    checks.append(("180 of the 256 products of 4-bit numbers do not fit in 4 bits; keeping only the low 4 bits turns 225 into 1",
                   int((P > 15).sum()) == 180 and int((P <= 15).sum()) == 76 and L.low_half(225, 4) == 1))
    a8, b8 = np.meshgrid(np.arange(256), np.arange(256))
    too_big = int(((a8 * b8) > 255).sum())
    checks.append(("8 x 8: 63568 of the 65536 products (97.0 %) do not fit in 8 bits", too_big == 63568 and round(100 * too_big / 65536, 1) == 97.0))

    # design budgets
    checks.append(("widest multiplier within 20 gate delays (10 ns) is 4 bits (16 gate delays; 5 bits needs 22); within 40 gate delays (20 ns) it is 8 bits (9 bits needs 46)",
                   L.widest_within(20) == 4 and L.design_row(5)["depth"] == 22 and L.widest_within(40) == 8
                   and L.design_row(9)["depth"] == 46))

    # mini-project: 8 x 8 multiplier
    m8 = L.build_multiplier(8)
    p8 = L.multiply(m8, a8.ravel(), b8.ravel())
    checks.append(("mini-project: the 8 x 8 multiplier matches Python for all 65536 pairs and has 320 gates, 16 product bits and a slowest bit after 40 gate delays (20 ns)",
                   np.array_equal(p8, (a8 * b8).ravel()) and m8["c"].size() == 320 and len(m8["product"]) == 16
                   and max(L.product_delays(m8)) == 40 and L.time_ns(40) == 20.0))
    checks.append(("mini-project: the largest 8 x 8 product is 255 x 255 = 65025 = 1111111000000001, which needs all 16 bits",
                   int(p8.max()) == 65025 == 0b1111111000000001))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
